## UrbanRide_12 — Fraud Detection Model
**Type:** Binary Classification  
**Label:** `is_suspicious` (true = suspicious, false = normal)  
**Source:** `urbanride.gold.payment_features`  
**Schedule:** Weekly · Sunday 04:00 AM (Job 4)

**Models trained:**
- Logistic Regression — baseline
- Random Forest — main model (numTrees=50, maxDepth=10 — confirmed safe from Churn model)

**Key challenges:**
- 19.6M rows — too large for serverless 1GB ML cache → train on 20% stratified sample
- 0.74% suspicious — severe class imbalance → handled via `weightCol`
- Orphan payments have NULL trip context → `handleInvalid='skip'` in VectorAssembler

**Strongest fraud signals (from Gold analysis):**
- `fare_delta_pct` — 0.0 normal vs 0.0795 suspicious
- `is_fraud_card`, `is_orphan_payment`, `is_fare_mismatch` — boolean flags
- `amount` — suspicious payments 10% higher

**Experiment:** `/urbanride_fraud_detection`  
**Registry:** `urbanride.default.urbanride_fraud_model`  
**Output:** `urbanride.gold.fraud_predictions`

In [0]:
# ── Run this cell first before anything else ─────────────────
# Clears stale ML models from previous sessions
# Prevents 1GB ML cache overflow error on serverless

import gc

stale_models = [
    'lr_model', 'rf_model', 'best_rf_model',
    'BEST_MODEL', 'rf_pipeline', 'lr_pipeline'
]
for name in stale_models:
    try:
        del globals()[name]
        print(f'  Deleted: {name}')
    except KeyError:
        pass

gc.collect()
print('  Python memory cleared.')
print()

spark.sql('SELECT 1').show()
print('  Spark connection healthy.')

spark.sql('SHOW SCHEMAS IN urbanride').show()
print('  Catalog accessible.')

print()
print('Session ready. Safe to run notebook.')


  Python memory cleared.

+---+
|  1|
+---+
|  1|
+---+

  Spark connection healthy.
+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+

  Catalog accessible.

Session ready. Safe to run notebook.


In [0]:
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, lit, current_timestamp
import pandas as pd
import gc
import tempfile
import os

# Do NOT call mlflow.autolog() — crashes on serverless

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
EXPERIMENT_NAME = '/urbanride_fraud_detection'
MODEL_NAME      = 'urbanride.default.urbanride_fraud_model'

# TMP_DIR — MLflow staging area during model save
# PREREQUISITE: Volume 'mlflow_artifacts' must exist in silver schema
TMP_DIR = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'

try:
    dbutils.fs.ls(TMP_DIR)
    print(f'TMP_DIR already exists: {TMP_DIR}')
except Exception:
    dbutils.fs.mkdirs(TMP_DIR)
    print(f'TMP_DIR created: {TMP_DIR}')

# Sample fraction — 19.6M rows too large for serverless 1GB ML cache
# 20% stratified sample = ~3.9M rows — safe and representative
SAMPLE_FRACTION = 0.20
SAMPLE_SEED     = 42

# Feature columns
# Numeric — includes boolean flags cast to int
# NUMERIC_COLS = [
#     'amount',
#     'fare_delta_pct',
#     'trip_fare_amount',
#     'distance_km',
#     'fare_per_km',
#     'surge_multiplier',
#     'driver_rating',
#     'is_fraud_card',
#     'is_orphan_payment',
#     'is_fare_mismatch',
#     'is_ghost_trip',
#     'is_peak_surge',
#     'is_high_value_driver',
#     'is_distance_invalid',
# ]

NUMERIC_COLS = [
    'amount',
    'fare_delta_pct',
    'trip_fare_amount',
    'distance_km',
    'fare_per_km',
    'surge_multiplier',
    'driver_rating',
    'is_ghost_trip',
    'is_peak_surge',
    'is_high_value_driver',
    'is_distance_invalid',
]

CATEGORICAL_COLS = [
    'payment_method',
    'payment_status',
    'vehicle_type',
    'weather',
    'day_type',
    'city',
]

LABEL_COL  = 'label'
WEIGHT_COL = 'class_weight'

mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Catalog             : {CATALOG}')
print(f'Source              : {GOLD}.payment_features')
print(f'Experiment          : {EXPERIMENT_NAME}')
print(f'Model name          : {MODEL_NAME}')
print(f'Sample fraction     : {SAMPLE_FRACTION*100:.0f}%')
print(f'Numeric features    : {len(NUMERIC_COLS)}')
print(f'Categorical features: {len(CATEGORICAL_COLS)}')
print('Config ready.')


TMP_DIR already exists: /Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp
Catalog             : urbanride
Source              : urbanride.gold.payment_features
Experiment          : /urbanride_fraud_detection
Model name          : urbanride.default.urbanride_fraud_model
Sample fraction     : 20%
Numeric features    : 11
Categorical features: 6
Config ready.


In [0]:
print('Loading gold.payment_features...')

df_raw = spark.table(f'{GOLD}.payment_features')
total_count = df_raw.count()
print(f'  Total rows : {total_count:,}')

# Suspicious distribution
print()
print('Suspicious distribution (full dataset):')
df_raw.groupBy('is_suspicious').count().orderBy('is_suspicious').show()

# Select needed columns
# Cast boolean flags to int — Spark ML requires numeric features
# Cast boolean label to int
ALL_FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

df = df_raw.select(
    ['payment_id', 'customer_id', 'payment_date', 'is_suspicious'] + ALL_FEATURE_COLS
).withColumn(
    LABEL_COL, col('is_suspicious').cast('int')
).withColumn(
    'is_ghost_trip', col('is_ghost_trip').cast('int')
).withColumn(
    'is_peak_surge', col('is_peak_surge').cast('int')
).withColumn(
    'is_high_value_driver',  col('is_high_value_driver').cast('int')
).withColumn(
    'is_distance_invalid', col('is_distance_invalid').cast('int')
)

# Drop rows with null label
df = df.filter(col(LABEL_COL).isNotNull())

# ── Stratified sampling — keep fraud ratio balanced ───────────
# Cannot just sample randomly — 0.74% fraud would become even rarer
# Sample suspicious and non-suspicious separately, then union
# This preserves the original fraud ratio in the sample
print(f'\nSampling {SAMPLE_FRACTION*100:.0f}% stratified by is_suspicious...')

df_suspicious     = df.filter(col(LABEL_COL) == 1).sample(SAMPLE_FRACTION, seed=SAMPLE_SEED)
df_not_suspicious = df.filter(col(LABEL_COL) == 0).sample(SAMPLE_FRACTION, seed=SAMPLE_SEED)
df_sample         = df_suspicious.union(df_not_suspicious)

sample_count = df_sample.count()
print(f'  Sample rows : {sample_count:,}')
print()
print('Suspicious distribution (sample):')
df_sample.groupBy('is_suspicious').count().orderBy('is_suspicious').show()


Loading gold.payment_features...
  Total rows : 19,600,753

Suspicious distribution (full dataset):
+-------------+--------+
|is_suspicious|   count|
+-------------+--------+
|        false|19455717|
|         true|  145036|
+-------------+--------+


Sampling 20% stratified by is_suspicious...
  Sample rows : 3,921,527

Suspicious distribution (sample):
+-------------+-------+
|is_suspicious|  count|
+-------------+-------+
|        false|3892416|
|         true|  29111|
+-------------+-------+



In [0]:
# Time-based split on payment_date
# Train : payments before Jan 1 2026  (~67%)
# Test  : payments on or after Jan 1 2026 (~33%)
# Same split date as Churn model — consistent across all ML notebooks

SPLIT_DATE = '2026-01-01'

df_train = df_sample.filter(col('payment_date') <  SPLIT_DATE)
df_test  = df_sample.filter(col('payment_date') >= SPLIT_DATE)

train_count = df_train.count()
test_count  = df_test.count()
total       = train_count + test_count

print(f'Split date : {SPLIT_DATE}')
print(f'Train rows : {train_count:,} ({train_count/total*100:.1f}%)')
print(f'Test rows  : {test_count:,}  ({test_count/total*100:.1f}%)')
print()

print('Suspicious rate in train set:')
df_train.groupBy('is_suspicious').count().orderBy('is_suspicious').show()
print('Suspicious rate in test set:')
df_test.groupBy('is_suspicious').count().orderBy('is_suspicious').show()


Split date : 2026-01-01
Train rows : 2,631,777 (67.1%)
Test rows  : 1,289,750  (32.9%)

Suspicious rate in train set:
+-------------+-------+
|is_suspicious|  count|
+-------------+-------+
|        false|2612237|
|         true|  19540|
+-------------+-------+

Suspicious rate in test set:
+-------------+-------+
|is_suspicious|  count|
+-------------+-------+
|        false|1280179|
|         true|   9571|
+-------------+-------+



In [0]:
# Class imbalance: ~0.74% suspicious vs ~99.26% normal
# Much more severe than churn (12% vs 88%)
# Without class weights model predicts 'normal' for everything
# and gets 99.26% accuracy while catching zero fraud

print('Computing class weights...')

train_total        = df_train.count()
suspicious_count   = df_train.filter(col(LABEL_COL) == 1).count()
normal_count       = df_train.filter(col(LABEL_COL) == 0).count()
num_classes        = 2

weight_suspicious  = train_total / (num_classes * suspicious_count)
weight_normal      = train_total / (num_classes * normal_count)

print(f'  Train total        : {train_total:,}')
print(f'  Suspicious count   : {suspicious_count:,}')
print(f'  Normal count       : {normal_count:,}')
print(f'  Weight suspicious  : {weight_suspicious:.4f}')
print(f'  Weight normal      : {weight_normal:.4f}')

df_train = df_train.withColumn(
    WEIGHT_COL,
    when(col(LABEL_COL) == 1, weight_suspicious)
    .otherwise(weight_normal)
)

df_test = df_test.withColumn(WEIGHT_COL, lit(1.0))

print('Class weights added to train set.')


Computing class weights...
  Train total        : 2,631,777
  Suspicious count   : 19,540
  Normal count       : 2,612,237
  Weight suspicious  : 67.3433
  Weight normal      : 0.5037
Class weights added to train set.


In [0]:
print('Building feature pipeline...')

# StringIndexer — encode categorical columns to numbers
# handleInvalid='keep' — orphan payments may have unseen values in test set
indexers = [
    StringIndexer(
        inputCol     = c,
        outputCol    = f'{c}_idx',
        handleInvalid= 'keep'
    )
    for c in CATEGORICAL_COLS
]

indexed_cat_cols = [f'{c}_idx' for c in CATEGORICAL_COLS]
ALL_INPUT_COLS   = NUMERIC_COLS + indexed_cat_cols

# VectorAssembler — combine all features into one vector
# handleInvalid='skip' — orphan payments have NULL trip context
# (driver_rating, surge_multiplier, city NULL for orphan payments)
# skip drops rows with nulls during training — safe for our use case
assembler = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features_raw',
    handleInvalid= 'skip'
)

# StandardScaler — needed for Logistic Regression only
# amount (~143) vs fare_delta_pct (~0.08) — very different ranges
scaler = StandardScaler(
    inputCol = 'features_raw',
    outputCol= 'features',
    withMean = True,
    withStd  = True
)

print(f'  Numeric features    : {len(NUMERIC_COLS)}')
print(f'  Categorical features: {len(CATEGORICAL_COLS)} → {len(indexed_cat_cols)} indexed')
print(f'  Total input cols    : {len(ALL_INPUT_COLS)}')
print('Feature pipeline ready.')


Building feature pipeline...
  Numeric features    : 11
  Categorical features: 6 → 6 indexed
  Total input cols    : 17
Feature pipeline ready.


In [0]:
print('Training Logistic Regression (baseline)...')

lr = LogisticRegression(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    weightCol   = WEIGHT_COL,
    maxIter     = 20,
    regParam    = 0.01,
)

lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

auc_evaluator = BinaryClassificationEvaluator(
    labelCol   = LABEL_COL,
    metricName = 'areaUnderROC'
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol   = LABEL_COL,
    metricName = 'f1'
)

with mlflow.start_run(run_name='logistic_regression') as run:
    lr_run_id = run.info.run_id

    lr_model = lr_pipeline.fit(df_train)
    lr_preds = lr_model.transform(df_test)

    lr_auc = auc_evaluator.evaluate(lr_preds)
    lr_f1  = f1_evaluator.evaluate(lr_preds)

    mlflow.log_param('model_type',     'LogisticRegression')
    mlflow.log_param('max_iter',       20)
    mlflow.log_param('reg_param',      0.01)
    mlflow.log_param('split_date',     SPLIT_DATE)
    mlflow.log_param('sample_fraction',SAMPLE_FRACTION)
    mlflow.log_param('train_rows',     df_train.count())
    mlflow.log_param('test_rows',      df_test.count())
    mlflow.log_metric('auc', lr_auc)
    mlflow.log_metric('f1',  lr_f1)

print(f'  LR AUC : {lr_auc:.4f}')
print(f'  LR F1  : {lr_f1:.4f}')


Training Logistic Regression (baseline)...
  LR AUC : 0.6883
  LR F1  : 0.9966


In [0]:
print('Training Random Forest...')
print()

# Clear LR model output and pipeline from cache before fitting RF

if 'lr_pipeline' in globals():
    del lr_pipeline
if 'rf_pipeline' in globals():
    del rf_pipeline
if 'lr_preds' in globals():
    del lr_preds
import gc
gc.collect()
print('LR model and outputs cleared from cache.')

# ── Why single combo? ─────────────────────────────────────────
# Serverless 1GB ML cache limit — confirmed from Churn model run
# numTrees=50, maxDepth=10 confirmed safe and gave AUC=0.9927 on churn
# RF does NOT need StandardScaler — trees are scale-invariant
# ─────────────────────────────────────────────────────────────

assembler_rf = VectorAssembler(
    inputCols    = ALL_INPUT_COLS,
    outputCol    = 'features',
    handleInvalid= 'skip'
)

best_rf_params = {'numTrees': 50, 'maxDepth': 10}

rf = RandomForestClassifier(
    featuresCol = 'features',
    labelCol    = LABEL_COL,
    weightCol   = WEIGHT_COL,
    numTrees    = best_rf_params['numTrees'],
    maxDepth    = best_rf_params['maxDepth'],
    seed        = 42,
)

rf_pipeline = Pipeline(stages=indexers + [assembler_rf, rf])

with mlflow.start_run(run_name='rf_trees50_depth10') as run:
    best_rf_model = rf_pipeline.fit(df_train)
    best_rf_preds = best_rf_model.transform(df_test)

    best_rf_auc = auc_evaluator.evaluate(best_rf_preds)
    best_rf_f1  = f1_evaluator.evaluate(best_rf_preds)

    mlflow.log_param('model_type', 'RandomForest')
    mlflow.log_param('num_trees', best_rf_params['numTrees'])
    mlflow.log_param('max_depth', best_rf_params['maxDepth'])
    mlflow.log_param('split_date', SPLIT_DATE)
    mlflow.log_param('sample_fraction',SAMPLE_FRACTION)
    mlflow.log_metric('auc', best_rf_auc)
    mlflow.log_metric('f1',  best_rf_f1)
    best_rf_run_id = run.info.run_id

print(f'  RF params : {best_rf_params}')
print(f'  RF AUC    : {best_rf_auc:.4f}')
print(f'  RF F1     : {best_rf_f1:.4f}')
print('  best_rf_model ready for Cell 8.')


Training Random Forest...

LR model and outputs cleared from cache.
  RF params : {'numTrees': 50, 'maxDepth': 10}
  RF AUC    : 0.6989
  RF F1     : 0.9966
  best_rf_model ready for Cell 8.


In [0]:
print('='*55)
print('FRAUD MODEL COMPARISON')
print('='*55)
print(f'{"Model":<40} {"AUC":>8} {"F1":>8}')
print('-'*55)
print(f'{"Logistic Regression":<40} {lr_auc:>8.4f} {lr_f1:>8.4f}')
print(f'{"Random Forest (numTrees=50, maxDepth=10)":<40} {best_rf_auc:>8.4f} {best_rf_f1:>8.4f}')
print('='*55)

# Pick best by AUC
if best_rf_auc >= lr_auc:
    BEST_MODEL      = best_rf_model
    BEST_RUN_ID     = best_rf_run_id
    BEST_AUC        = best_rf_auc
    BEST_MODEL_TYPE = f'RandomForest_{best_rf_params}'
    print('\nWinner: Random Forest')
else:
    BEST_MODEL      = lr_model
    BEST_RUN_ID     = lr_run_id
    BEST_AUC        = lr_auc
    BEST_MODEL_TYPE = 'LogisticRegression'
    print('\nWinner: Logistic Regression')

print(f'Best AUC    : {BEST_AUC:.4f}')
print(f'Best run ID : {BEST_RUN_ID}')
print(f'\nCheck Experiments UI: {EXPERIMENT_NAME}')


FRAUD MODEL COMPARISON
Model                                         AUC       F1
-------------------------------------------------------
Logistic Regression                        0.6883   0.9966
Random Forest (numTrees=50, maxDepth=10)   0.6989   0.9966

Winner: Random Forest
Best AUC    : 0.6989
Best run ID : 6044560d18324233a826de7ec7809d37

Check Experiments UI: /urbanride_fraud_detection


In [0]:
print('Registering best model to MLflow Model Registry...')

client = MlflowClient()

# Step 1 — Create model name in registry
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = 'Detects suspicious payments for ZipCab ride-hailing platform.'
    )
    print(f'  Created model: {MODEL_NAME}')
except Exception:
    print(f'  Model already exists: {MODEL_NAME} — adding new version')

# Step 2 — Build model signature
sample_input  = df_test.select(ALL_FEATURE_COLS).limit(100)
sample_pandas = sample_input.toPandas()

sample_preds  = BEST_MODEL.transform(sample_input)
sample_output = (
    sample_preds
    .withColumn('fraud_probability', vector_to_array('probability')[1])
    .select('prediction', 'fraud_probability')
    .toPandas()
)

signature = infer_signature(
    model_input  = sample_pandas,
    model_output = sample_output
)
print('  Signature inferred.')
print(signature)

# Step 3 — Log model + register
with mlflow.start_run(run_name='register_fraud_model') as reg_run:
    mlflow.log_param('model_type',     BEST_MODEL_TYPE)
    mlflow.log_param('best_auc',       BEST_AUC)
    mlflow.log_param('sample_fraction',SAMPLE_FRACTION)
    mlflow.log_param('feature_count',  len(ALL_INPUT_COLS))
    mlflow.log_metric('auc', BEST_AUC)

    # Feature importance artifact (RF only)
    if 'RandomForest' in BEST_MODEL_TYPE:
        rf_stage = BEST_MODEL.stages[-1]
        fi_df = pd.DataFrame({
            'feature':    ALL_INPUT_COLS,
            'importance': rf_stage.featureImportances.toArray()
        }).sort_values('importance', ascending=False)
        with tempfile.TemporaryDirectory() as tmp:
            fi_path = os.path.join(tmp, 'fraud_feature_importance.csv')
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)
        print('  Feature importance logged.')

    mlflow.spark.log_model(
        spark_model   = BEST_MODEL,
        artifact_path = 'fraud_model',
        signature     = signature,
        input_example = sample_pandas.head(5),
        dfs_tmpdir    = TMP_DIR
    )

    model_uri = f'runs:/{reg_run.info.run_id}/fraud_model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)

# Step 4 — Add version description
versions       = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max([int(v.version) for v in versions])

client.update_model_version(
    name        = MODEL_NAME,
    version     = str(latest_version),
    description = (
        f'{BEST_MODEL_TYPE}. AUC={BEST_AUC:.4f}. '
        f'Trained on 20% stratified sample of ZipCab payments. '
        f'Split: trained before {SPLIT_DATE}. '
        f'Key feature: fare_delta_pct.'
    )
)
print(f'  Description added to {MODEL_NAME} v{latest_version}')

print()
print(f'  Model registered : {MODEL_NAME}')
print(f'  Version          : {latest_version}')
print(f'  Load URI         : models:/{MODEL_NAME}/{latest_version}')
print(f'  Check Models UI  : {MODEL_NAME}')


Registering best model to MLflow Model Registry...
  Created model: urbanride.default.urbanride_fraud_model
  Signature inferred.
inputs: 
  ['amount': double (required), 'fare_delta_pct': double (optional), 'trip_fare_amount': double (optional), 'distance_km': double (optional), 'fare_per_km': double (optional), 'surge_multiplier': double (optional), 'driver_rating': double (optional), 'is_ghost_trip': double (optional), 'is_peak_surge': double (optional), 'is_high_value_driver': double (optional), 'is_distance_invalid': double (optional), 'payment_method': string (required), 'payment_status': string (required), 'vehicle_type': string (optional), 'weather': string (optional), 'day_type': string (optional), 'city': string (optional)]
outputs: 
  ['prediction': double (required), 'fraud_probability': double (required)]
params: 
  None

  Feature importance logged.


2026/03/12 10:55:44 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/12 10:55:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-dce055e7-6dd4-4396-b0d7-f7/tmpzacn9fym/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/12 10:55:48 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "amount",
      "fare_delta_pct",
      "trip_fare_amount",
      "distance_km",
      "fare_per_km",
      "surge_multip

  Description added to urbanride.default.urbanride_fraud_model v1

  Model registered : urbanride.default.urbanride_fraud_model
  Version          : 1
  Load URI         : models:/urbanride.default.urbanride_fraud_model/1
  Check Models UI  : urbanride.default.urbanride_fraud_model


In [0]:
print('Saving fraud predictions to Delta table...')

# Score full dataset — not just sample
# Model trained on sample but we score all 19.6M payments
# Add WEIGHT_COL=1.0 as required by pipeline
df_all    = df.withColumn(WEIGHT_COL, lit(1.0))
df_scored = BEST_MODEL.transform(df_all)

df_predictions = df_scored.select(
    col('payment_id'),
    col('customer_id'),
    col('payment_date'),
    col('is_suspicious'),
    col(LABEL_COL).alias('actual_label'),
    col('prediction').alias('predicted_label'),
    vector_to_array('probability')[1].alias('fraud_probability'),

    # Risk tier — actionable for fraud team
    when(vector_to_array('probability')[1] >= 0.7, 'high_risk')
    .when(vector_to_array('probability')[1] >= 0.4, 'medium_risk')
    .otherwise('low_risk').alias('fraud_risk_tier'),

).withColumn('_processed_at',  current_timestamp()) \
 .withColumn('_model_version', lit(str(latest_version))) \
 .withColumn('_model_name',    lit(MODEL_NAME))

# Write to Gold layer
df_predictions.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.fraud_predictions')

pred_count = df_predictions.count()
print(f'  Predictions written : {pred_count:,}')
print(f'  Table               : {GOLD}.fraud_predictions')


Saving fraud predictions to Delta table...
  Predictions written : 19,444,155
  Table               : urbanride.gold.fraud_predictions


import mlflow.spark

TMP_DIR    = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'
MODEL_NAME = 'urbanride.default.urbanride_fraud_model'

BEST_MODEL = mlflow.spark.load_model(
    f'models:/{MODEL_NAME}/1',
    dfs_tmpdir = TMP_DIR
)
print('Model loaded.')

In [0]:
print('=== FRAUD MODEL VERIFICATION ===')
print()

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
BEST_MODEL_TYPE = 'RandomForest'
BEST_AUC    = 0.6989



df_verify = spark.table(f'{GOLD}.fraud_predictions')
total     = df_verify.count()

print(f'  Total predictions : {total:,}')
print(f'  Model             : {BEST_MODEL_TYPE}')
print(f'  Best AUC          : {BEST_AUC:.4f}')
print()

# Risk tier distribution
print('Risk tier distribution:')
df_verify.groupBy('fraud_risk_tier').count().orderBy('count', ascending=False).show()

# Confusion matrix
print('Prediction vs actual (confusion matrix):')
df_verify.groupBy('actual_label', 'predicted_label') \
    .count().orderBy('actual_label', 'predicted_label').show()

# Key validation — suspicious payments should have much higher avg probability
print('Avg fraud probability by actual suspicious:')
from pyspark.sql.functions import avg, round as spark_round
df_verify.groupBy('is_suspicious').agg(
    spark_round(avg('fraud_probability'), 4).alias('avg_fraud_probability')
).orderBy('is_suspicious').show()

# Feature importance — RF only
# fare_delta_pct should be top feature — confirms Gold analysis
# if 'RandomForest' in BEST_MODEL_TYPE:
#     print('Top 10 most important features:')
#     rf_stage = BEST_MODEL.stages[-1]
#     fi = sorted(
#         zip(ALL_INPUT_COLS, rf_stage.featureImportances.toArray()),
#         key=lambda x: x[1],
#         reverse=True
#     )
#     print(f'  {"Feature":<35} {"Importance":>10}')
#     print(f'  {"-"*47}')
#     for feat, imp in fi[:10]:
#         print(f'  {feat:<35} {imp:>10.4f}')

# print()
# print('Sample high risk payments:')
# df_verify.filter(col('fraud_risk_tier') == 'high_risk').select(
#     'payment_id', 'is_suspicious',
#     'fraud_probability', 'fraud_risk_tier'
# ).limit(5).show(truncate=False)

print()
print('='*55)
print('Fraud model complete.')
# print(f'Experiments UI : {EXPERIMENT_NAME}')
print(f'Models UI      : {MODEL_NAME}')
print(f'Predictions    : {GOLD}.fraud_predictions')
print('Next           : UrbanRide_13 — Demand Forecasting Model')
print('='*55)


=== FRAUD MODEL VERIFICATION ===

  Total predictions : 19,444,155
  Model             : RandomForest
  Best AUC          : 0.6989

Risk tier distribution:
+---------------+--------+
|fraud_risk_tier|   count|
+---------------+--------+
|       low_risk|19410344|
|      high_risk|   33367|
|    medium_risk|     444|
+---------------+--------+

Prediction vs actual (confusion matrix):
+------------+---------------+--------+
|actual_label|predicted_label|   count|
+------------+---------------+--------+
|           0|            0.0|19357820|
|           0|            1.0|     261|
|           1|            0.0|   52684|
|           1|            1.0|   33390|
+------------+---------------+--------+

Avg fraud probability by actual suspicious:
+-------------+---------------------+
|is_suspicious|avg_fraud_probability|
+-------------+---------------------+
|        false|               0.2614|
|         true|               0.5383|
+-------------+---------------------+


Fraud model comple

### Result

- AUC = 0.6989 — weak but honest
- Caught only 38.8% of suspicious payments
- Without the rule-based flags no strong ML signal exists in remaining features
- Rule-based Gold layer flags remain the primary fraud detection mechanism
- ML model adds a secondary probabilistic layer on top